# Producto U1 — David Romero Nina

**Dimensión:** Dirección dominante del flujo (`down_up_ratio`) — U1 batch, clasificación (ángulo BI)

**Rol en el equipo:** BI / ML

**Curso:** Big Data · lambda26 · Proyecto Sello (equipo LLSW3, sección GU)

**Caso:** análisis de tráfico de red del campus universitario (dataset propio, ~400 000 flujos, capturado vía Suricata)

**Metodología:** CRISP-DM — Fases 1 a 5 (hasta modelado/evaluación; el despliegue es Unidad 2)


## Arquitectura Big Data (contexto — no es una fase de CRISP-DM)

Este notebook implementa la **ruta batch** de la arquitectura **Lambda** declarada en el
[Brief técnico-analítico](../../docs/proyecto-sello/brief.md) (Hito S2): capa batch (este
notebook) + capa de velocidad (Kafka, contenido de Unidad 2). El detalle completo de la
decisión Lambda vs. Kappa está en el brief.

## Fase 1 — Comprensión del negocio (CRISP-DM)

**Pregunta de negocio (dimensión propia):** ¿Qué proporción de los flujos históricos son de
descarga dominante, carga dominante o balanceados, y qué patrón se puede esperar?

**Objetivo de minería de datos:** entrenar un modelo de **clasificación** (3 categorías) que
prediga la dirección dominante del flujo a partir de variables de comportamiento (protocolo,
tiempos, banderas) — sin usar directamente las columnas de tamaño/bytes de las que se deriva
`down_up_ratio`.

**Decisión que habilita:** priorizar ancho de banda saliente vs. entrante y detectar cambios
inusuales en el patrón de uso del campus.

**Criterio de éxito:** al construir las 3 categorías por cuantiles (33/33/33 aprox.), la
línea base ingenua (predecir siempre la clase mayoritaria) ronda 33-34% de accuracy — el
modelo debe superarla con margen claro (referencia orientativa: accuracy > 50%).


## Fase 2 — Comprensión de los datos (CRISP-DM)

### Extracción con esquema explícito


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, DoubleType
)
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("u1-producto-david-direccion")
    .getOrCreate()
)

RUTA_DATOS = "/opt/data/TRCU.csv"

# Hallazgo documentado en S03: con header=True + StructType explícito, Spark asigna
# los campos por POSICION, no por nombre de columna. Si el orden de StructField no
# coincide exactamente con el orden físico del CSV, los valores se corrompen en
# silencio (sin lanzar error). Por eso el orden de abajo respeta el header real
# observado en el dataset, columna por columna.

esquema_flujos = StructType([
    StructField("flow_id", StringType(), True),
    StructField("src_addr", StringType(), True),
    StructField("src_port", IntegerType(), True),
    StructField("dst_addr", StringType(), True),
    StructField("dst_port", IntegerType(), True),
    StructField("ip_prot", IntegerType(), True),
    StructField("timestamp", LongType(), True),
    StructField("flow_duration", DoubleType(), True),
    StructField("down_up_ratio", DoubleType(), True),
    StructField("pkt_len_max", DoubleType(), True),
    StructField("pkt_len_min", DoubleType(), True),
    StructField("pkt_len_mean", DoubleType(), True),
    StructField("pkt_len_var", DoubleType(), True),
    StructField("pkt_len_std", DoubleType(), True),
    StructField("bytes_per_s", DoubleType(), True),
    StructField("pkt_per_s", DoubleType(), True),
    StructField("fwd_pkt_per_s", DoubleType(), True),
    StructField("bwd_pkt_per_s", DoubleType(), True),
    StructField("fwd_pkt_cnt", IntegerType(), True),
    StructField("fwd_pkt_len_tot", DoubleType(), True),
    StructField("fwd_pkt_len_max", DoubleType(), True),
    StructField("fwd_pkt_len_min", DoubleType(), True),
    StructField("fwd_pkt_len_mean", DoubleType(), True),
    StructField("fwd_pkt_len_std", DoubleType(), True),
    StructField("fwd_pkt_hdr_len_tot", IntegerType(), True),
    StructField("fwd_pkt_hdr_len_min", IntegerType(), True),
    StructField("fwd_non_empty_pkt_cnt", IntegerType(), True),
    StructField("bwd_pkt_cnt", IntegerType(), True),
    StructField("bwd_pkt_len_tot", DoubleType(), True),
    StructField("bwd_pkt_len_max", DoubleType(), True),
    StructField("bwd_pkt_len_min", DoubleType(), True),
    StructField("bwd_pkt_len_mean", DoubleType(), True),
    StructField("bwd_pkt_len_std", DoubleType(), True),
    StructField("bwd_pkt_hdr_len_tot", IntegerType(), True),
    StructField("bwd_pkt_hdr_len_min", IntegerType(), True),
    StructField("bwd_non_empty_pkt_cnt", IntegerType(), True),
    StructField("iat_max", DoubleType(), True),
    StructField("iat_min", DoubleType(), True),
    StructField("iat_mean", DoubleType(), True),
    StructField("iat_std", DoubleType(), True),
    StructField("fwd_iat_tot", DoubleType(), True),
    StructField("fwd_iat_max", DoubleType(), True),
    StructField("fwd_iat_min", DoubleType(), True),
    StructField("fwd_iat_mean", DoubleType(), True),
    StructField("fwd_iat_std", DoubleType(), True),
    StructField("bwd_iat_tot", DoubleType(), True),
    StructField("bwd_iat_max", DoubleType(), True),
    StructField("bwd_iat_min", DoubleType(), True),
    StructField("bwd_iat_mean", DoubleType(), True),
    StructField("bwd_iat_std", DoubleType(), True),
    StructField("active_max", DoubleType(), True),
    StructField("active_min", DoubleType(), True),
    StructField("active_mean", DoubleType(), True),
    StructField("active_std", DoubleType(), True),
    StructField("idle_max", DoubleType(), True),
    StructField("idle_min", DoubleType(), True),
    StructField("idle_mean", DoubleType(), True),
    StructField("idle_std", DoubleType(), True),
    StructField("flag_SYN", IntegerType(), True),
    StructField("flag_fin", IntegerType(), True),
    StructField("flag_rst", IntegerType(), True),
    StructField("flag_ack", IntegerType(), True),
    StructField("flag_psh", IntegerType(), True),
    StructField("fwd_flag_psh", IntegerType(), True),
    StructField("bwd_flag_psh", IntegerType(), True),
    StructField("flag_urg", IntegerType(), True),
    StructField("fwd_flag_urg", IntegerType(), True),
    StructField("bwd_flag_urg", IntegerType(), True),
    StructField("flag_cwr", IntegerType(), True),
    StructField("flag_ece", IntegerType(), True),
    StructField("fwd_bulk_bytes_mean", DoubleType(), True),
    StructField("fwd_bulk_pkt_mean", DoubleType(), True),
    StructField("fwd_bulk_rate_mean", DoubleType(), True),
    StructField("bwd_bulk_bytes_mean", DoubleType(), True),
    StructField("bwd_bulk_pkt_mean", DoubleType(), True),
    StructField("bwd_bulk_rate_mean", DoubleType(), True),
    StructField("fwd_subflow_bytes_mean", DoubleType(), True),
    StructField("fwd_subflow_pkt_mean", DoubleType(), True),
    StructField("bwd_subflow_bytes_mean", DoubleType(), True),
    StructField("bwd_subflow_pkt_mean", DoubleType(), True),
    StructField("fwd_tcp_init_win_bytes", IntegerType(), True),
    StructField("bwd_tcp_init_win_bytes", IntegerType(), True),
    StructField("label", StringType(), True),
])

df = spark.read.csv(RUTA_DATOS, header=True, schema=esquema_flujos)

with open(RUTA_DATOS, "r", encoding="utf-8") as f:
    cabecera_real = f.readline().strip().split(",")
assert cabecera_real == [c.name for c in esquema_flujos.fields], (
    "El orden del esquema no coincide con el header real del CSV — revisar antes de continuar."
)

df.printSchema()
df.show(5, truncate=False)
print("Filas totales:", df.count())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/11 03:48:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- flow_id: string (nullable = true)
 |-- src_addr: string (nullable = true)
 |-- src_port: integer (nullable = true)
 |-- dst_addr: string (nullable = true)
 |-- dst_port: integer (nullable = true)
 |-- ip_prot: integer (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- flow_duration: double (nullable = true)
 |-- down_up_ratio: double (nullable = true)
 |-- pkt_len_max: double (nullable = true)
 |-- pkt_len_min: double (nullable = true)
 |-- pkt_len_mean: double (nullable = true)
 |-- pkt_len_var: double (nullable = true)
 |-- pkt_len_std: double (nullable = true)
 |-- bytes_per_s: double (nullable = true)
 |-- pkt_per_s: double (nullable = true)
 |-- fwd_pkt_per_s: double (nullable = true)
 |-- bwd_pkt_per_s: double (nullable = true)
 |-- fwd_pkt_cnt: integer (nullable = true)
 |-- fwd_pkt_len_tot: double (nullable = true)
 |-- fwd_pkt_len_max: double (nullable = true)
 |-- fwd_pkt_len_min: double (nullable = true)
 |-- fwd_pkt_len_mean: double (nullable = true)
 |

26/09/11 03:48:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------------------------------------+--------------+--------+--------------+--------+-------+----------------+-------------+-------------+-----------+-----------+------------+-------------+-----------+------------+---------+-------------+-------------+-----------+---------------+---------------+---------------+----------------+---------------+-------------------+-------------------+---------------------+-----------+---------------+---------------+---------------+----------------+---------------+-------------------+-------------------+---------------------+--------+-------+------------+-------------+-----------+-----------+-----------+------------+-------------+-----------+-----------+-----------+------------+-------------+----------+----------+-----------+----------+--------+--------+---------+--------+--------+--------+--------+--------+--------+------------+------------+--------+------------+------------+--------+--------+-------------------+-----------------+------------------

Filas totales: 397354


### Exploración inicial (EDA)


In [2]:
print("Resumen estadistico de down_up_ratio:")
df.select("down_up_ratio").describe().show()

percentiles = df.approxQuantile("down_up_ratio", [0.0, 0.33, 0.5, 0.66, 1.0], 0.01)
print("down_up_ratio -> min/p33/mediana/p66/max:", percentiles)

print("Nulos en down_up_ratio:", df.filter(F.col("down_up_ratio").isNull()).count())

print("Distribucion por protocolo (ip_prot):")
df.groupBy("ip_prot").count().orderBy(F.desc("count")).show()


Resumen estadistico de down_up_ratio:


+-------+--------------------+
|summary|       down_up_ratio|
+-------+--------------------+
|  count|              397354|
|   mean|0.001661095871691...|
| stddev|  0.0445320584427454|
|    min|                 0.0|
|    max|                 2.4|
+-------+--------------------+



down_up_ratio -> min/p33/mediana/p66/max: [0.0, 0.0, 0.0, 0.0, 2.4]


Nulos en down_up_ratio: 0
Distribucion por protocolo (ip_prot):


+-------+------+
|ip_prot| count|
+-------+------+
|     17|374483|
|      6| 22579|
|      2|   243|
|      1|    49|
+-------+------+



## Fase 3 — Preparación de los datos (CRISP-DM)

### Transformación y agregación


In [3]:
cuantiles = df.approxQuantile("down_up_ratio", [0.33, 0.66], 0.01)
p33, p66 = cuantiles[0], cuantiles[1]
print("Cortes calculados sobre el historico -> p33:", p33, " p66:", p66)

df_david = df.withColumn(
    "categoria_direccion",
    F.when(F.col("down_up_ratio") < p33, F.lit("carga_dominante"))
     .when(F.col("down_up_ratio") > p66, F.lit("descarga_dominante"))
     .otherwise(F.lit("balanceado"))
)

df_david.explain(True)

distribucion = df_david.groupBy("categoria_direccion").agg(F.count("*").alias("n_flujos"))
distribucion.show()


Cortes calculados sobre el historico -> p33: 0.0  p66: 0.0
== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(categoria_direccion, CASE WHEN '`<`('down_up_ratio, 0.0) THEN carga_dominante WHEN '`>`('down_up_ratio, 0.0) THEN descarga_dominante ELSE balanceado END, None)]
+- Relation [flow_id#0,src_addr#1,src_port#2,dst_addr#3,dst_port#4,ip_prot#5,timestamp#6L,flow_duration#7,down_up_ratio#8,pkt_len_max#9,pkt_len_min#10,pkt_len_mean#11,pkt_len_var#12,pkt_len_std#13,bytes_per_s#14,pkt_per_s#15,fwd_pkt_per_s#16,bwd_pkt_per_s#17,fwd_pkt_cnt#18,fwd_pkt_len_tot#19,fwd_pkt_len_max#20,fwd_pkt_len_min#21,fwd_pkt_len_mean#22,fwd_pkt_len_std#23,fwd_pkt_hdr_len_tot#24,... 58 more fields] csv

== Analyzed Logical Plan ==
flow_id: string, src_addr: string, src_port: int, dst_addr: string, dst_port: int, ip_prot: int, timestamp: bigint, flow_duration: double, down_up_ratio: double, pkt_len_max: double, pkt_len_min: double, pkt_len_mean: double, pkt_len_var: double, pkt_len_std: double, byte

+-------------------+--------+
|categoria_direccion|n_flujos|
+-------------------+--------+
|         balanceado|  396747|
| descarga_dominante|     607|
+-------------------+--------+



### Calidad de datos y particionamiento analítico


In [4]:
df_dedup = df_david.dropDuplicates(["flow_id"])
df_limpio = df_dedup.na.drop(subset=["down_up_ratio", "categoria_direccion"])

RUTA_SALIDA = "/opt/artifacts/david/flujos_particionado"
(
    df_limpio.write.mode("overwrite")
    .partitionBy("categoria_direccion")
    .parquet(RUTA_SALIDA)
)

df_verificacion = spark.read.parquet(RUTA_SALIDA)
print("Filas tras limpieza:", df_limpio.count())
print("Filas leidas de vuelta desde Parquet:", df_verificacion.count())

df_verificacion.filter(F.col("categoria_direccion") == "descarga_dominante").explain(True)

26/09/11 03:48:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/11 03:48:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/11 03:48:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/11 03:48:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/09/11 03:48:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/09/11 03:48:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/09/11 03:48:53 WARN MemoryManager: Total allocation exceeds 95.

26/09/11 03:48:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 40.00% for 19 writers
26/09/11 03:48:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 42.22% for 18 writers
26/09/11 03:48:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 44.71% for 17 writers
26/09/11 03:48:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 47.50% for 16 writers
26/09/11 03:48:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 50.67% for 15 writers
26/09/11 03:48:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 54.29% for 14 writers
26/09/11 03:48:53 WARN MemoryManager: Total allocation exceeds 9

Filas tras limpieza: 133311


Filas leidas de vuelta desde Parquet: 133311
== Parsed Logical Plan ==
'Filter '`=`('categoria_direccion, descarga_dominante)
+- Relation [flow_id#1819,src_addr#1820,src_port#1821,dst_addr#1822,dst_port#1823,ip_prot#1824,timestamp#1825L,flow_duration#1826,down_up_ratio#1827,pkt_len_max#1828,pkt_len_min#1829,pkt_len_mean#1830,pkt_len_var#1831,pkt_len_std#1832,bytes_per_s#1833,pkt_per_s#1834,fwd_pkt_per_s#1835,bwd_pkt_per_s#1836,fwd_pkt_cnt#1837,fwd_pkt_len_tot#1838,fwd_pkt_len_max#1839,fwd_pkt_len_min#1840,fwd_pkt_len_mean#1841,fwd_pkt_len_std#1842,fwd_pkt_hdr_len_tot#1843,... 59 more fields] parquet

== Analyzed Logical Plan ==
flow_id: string, src_addr: string, src_port: int, dst_addr: string, dst_port: int, ip_prot: int, timestamp: bigint, flow_duration: double, down_up_ratio: double, pkt_len_max: double, pkt_len_min: double, pkt_len_mean: double, pkt_len_var: double, pkt_len_std: double, bytes_per_s: double, pkt_per_s: double, fwd_pkt_per_s: double, bwd_pkt_per_s: double, fwd_pkt_cn

### Selección de predictores y ensamblado del vector de features


In [5]:
from pyspark.ml.feature import VectorAssembler, StringIndexer

# Se excluyen a proposito las columnas de tamano/bytes (pkt_len_*, *_pkt_len_tot,
# bytes_per_s, *_bulk_*) porque down_up_ratio se deriva directamente de ellas — usarlas
# como predictor seria fuga de informacion. Los predictores se limitan a protocolo,
# tiempos y banderas: comportamiento, no volumen.
predictores_david = [
    "ip_prot", "flow_duration", "iat_mean", "iat_std", "fwd_iat_mean", "bwd_iat_mean",
    "flag_SYN", "flag_ack", "flag_psh", "active_mean", "idle_mean",
    "fwd_tcp_init_win_bytes", "bwd_tcp_init_win_bytes",
]

indexador = StringIndexer(inputCol="categoria_direccion", outputCol="label_idx")
modelo_indexador = indexador.fit(df_limpio)
df_indexado = modelo_indexador.transform(df_limpio)

ensamblador = VectorAssembler(inputCols=predictores_david, outputCol="features", handleInvalid="skip")
dataset_ml = ensamblador.transform(df_indexado).select("features", "label_idx")

df_train, df_test = dataset_ml.randomSplit([0.8, 0.2], seed=42)
print("Filas de entrenamiento:", df_train.count(), " / Filas de prueba:", df_test.count())


Filas de entrenamiento: 106817  / Filas de prueba: 26494


## Fase 4 — Modelado (CRISP-DM)


In [6]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier

configuraciones_david = {
    "LogisticRegression base": LogisticRegression(featuresCol="features", labelCol="label_idx"),
    "LogisticRegression + regularizacion": LogisticRegression(featuresCol="features", labelCol="label_idx", regParam=0.1, elasticNetParam=0.5),
    "RandomForestClassifier": RandomForestClassifier(featuresCol="features", labelCol="label_idx", seed=42),
}

modelos_entrenados_david = {}
predicciones_david = {}
for nombre, estimador in configuraciones_david.items():
    modelo = estimador.fit(df_train)
    modelos_entrenados_david[nombre] = modelo
    predicciones_david[nombre] = modelo.transform(df_test)
    print("Entrenado:", nombre)


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


Entrenado: LogisticRegression base


Entrenado: LogisticRegression + regularizacion


Entrenado: RandomForestClassifier


## Fase 5 — Evaluación (CRISP-DM)


In [7]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

ev_acc = MulticlassClassificationEvaluator(labelCol="label_idx", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="label_idx", metricName="f1")
ev_prec = MulticlassClassificationEvaluator(labelCol="label_idx", metricName="weightedPrecision")

clase_mayoritaria = df_test.groupBy("label_idx").count().orderBy(F.desc("count")).first()
frac_mayoritaria = clase_mayoritaria["count"] / df_test.count()
print(f"Linea base ingenua (predecir siempre la clase mayoritaria) -> accuracy={frac_mayoritaria:.4f}\n")

resultados_david = {}
for nombre, pred in predicciones_david.items():
    acc = ev_acc.evaluate(pred)
    f1 = ev_f1.evaluate(pred)
    prec = ev_prec.evaluate(pred)
    resultados_david[nombre] = f1
    print(f"{nombre:38s} Accuracy={acc:.4f}  F1={f1:.4f}  Precision={prec:.4f}")

nombre_ganador = max(resultados_david, key=resultados_david.get)
print(f"\nModelo ganador (mayor F1 ponderado): {nombre_ganador}")

mejor_pred = predicciones_david[nombre_ganador]
mejor_pred.groupBy("label_idx", "prediction").count().orderBy("label_idx", "prediction").show(50)

modelo_ganador = modelos_entrenados_david[nombre_ganador]
modelo_ganador.write().overwrite().save("/opt/artifacts/david/modelo_direccion")
print("Modelo ganador guardado en /opt/artifacts/david/modelo_direccion")

Linea base ingenua (predecir siempre la clase mayoritaria) -> accuracy=0.9958



LogisticRegression base                Accuracy=0.9977  F1=0.9973  Precision=0.9977


LogisticRegression + regularizacion    Accuracy=0.9958  F1=0.9938  Precision=0.9917


RandomForestClassifier                 Accuracy=0.9978  F1=0.9974  Precision=0.9978

Modelo ganador (mayor F1 ponderado): RandomForestClassifier


+---------+----------+-----+
|label_idx|prediction|count|
+---------+----------+-----+
|      0.0|       0.0|26384|
|      1.0|       0.0|   59|
|      1.0|       1.0|   51|
+---------+----------+-----+



Modelo ganador guardado en /opt/artifacts/david/modelo_direccion


## Cierre de fases y alcance

Este notebook cubre las **5 fases de CRISP-DM hasta el modelado/evaluación**: Comprensión del
negocio → Comprensión de los datos → Preparación de los datos → Modelado → Evaluación.
**No incluye la Fase 6 (Despliegue):** poner el modelo a inferir sobre flujos en vivo es
contenido de Unidad 2 (Spark Structured Streaming + Kafka), declarado como dimensión U2 en
el brief.

## Hallazgo(s) de esta dimensión

Ejecutado de punta a punta contra `TRCU.csv` (397 354 flujos reales capturados por Suricata):

- **El supuesto de cuantiles 33/33/33 no se cumple con datos reales:** `down_up_ratio` está
  fuertemente concentrado en 0 (min=p25=mediana=p66=0.0, max=2.4), así que los cortes de
  cuantil calculados sobre el histórico dan **p33 = p66 = 0.0**. Eso colapsa las 3 categorías
  previstas en la práctica a 2: `balanceado` (down_up_ratio == 0) termina siendo el **99.85%**
  de los flujos (396 747 de 397 354) y `descarga_dominante` (down_up_ratio > 0) solo el 0.15%
  (607 flujos); `carga_dominante` (< 0) queda **vacía**, porque `down_up_ratio` nunca es
  negativo en este dataset. La discretización por cuantiles de la Fase 3, tal como está
  definida, no es robusta frente a una variable con esta distribución degenerada.
- **Consecuencia en la evaluación:** con esa composición, la línea base ingenua ya alcanza
  accuracy=0.9958 y el ganador (`RandomForestClassifier`) llega a 0.9978 — solo **+0.20 pp**
  sobre la base, aunque ambos superan trivialmente el criterio de éxito nominal de la Fase 1
  (accuracy > 50%, pensado para un reparto ~33/33/33 que en la práctica no ocurrió). La
  matriz de confusión es más informativa que el accuracy: de los 110 flujos reales de
  `descarga_dominante` en el conjunto de prueba, el modelo identificó 51 (~46%) y confundió
  59 con `balanceado` — mejor que azar, pero lejos de ser confiable como está planteado hoy.
- **Recomendación:** redefinir la categorización de dirección dominante (p. ej. separar
  explícitamente `down_up_ratio == 0` como su propia categoría en vez de fusionarla con
  "balanceado", o recalibrar los cortes sobre la subpoblación con `down_up_ratio > 0`) antes
  de usar esta dimensión como base de la inferencia en vivo de Unidad 2.

## Cómo ejecutar este notebook (evidencia de contribución)

1. Levantar el laboratorio (`docker compose up -d` desde `pyspark/`).
2. El dataset real (`TRCU.csv`) ya está en `pyspark/data/`, montado en `/opt/data/` dentro del contenedor — no requiere ajustar `RUTA_DATOS`.
3. Ejecutar de punta a punta (`Run All`), sin intervención manual: el modelo ganador se elige y se guarda automáticamente en la Fase 5.
4. Confirmar la carpeta de salida (`!ls -R` o `os.walk`) sobre `/opt/artifacts/david/` (Parquet particionado + modelo guardado).
5. Capturar pantalla con reloj del sistema y usuario/perfil visibles.
6. Commit del notebook ejecutado al repositorio del equipo (`pyspark/artifacts/` no se versiona, ver `.gitignore`).